In [1]:
pip install nltk beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import urllib.request
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, WordNetLemmatizer

In [2]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\PGCP-AI\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
word = input("Enter a word: ").strip()
word

In [ ]:
synsets = wordnet.synsets(word)

In [ ]:


# =====================================================================
# QUESTION 1
# =====================================================================
print("--- Question 1 ---")
word = input("Enter a word: ").strip()

# 1. Print all meanings
print(f"\nAll meanings of '{word}':")
synsets = wordnet.synsets(word)
for i, syn in enumerate(synsets, 1):
    print(f"{i}. {syn.definition()}")

# 2. Print all noun meanings
print(f"\nNoun meanings of '{word}':")
noun_count = 1
for syn in synsets:
    if syn.pos() == 'n':  # 'n' stands for noun
        print(f"{noun_count}. {syn.definition()}")
        noun_count += 1

# =====================================================================
# QUESTION 2
# =====================================================================
print("\n--- Question 2 ---")

# Fetch and clean Wikipedia page content
url = "https://en.wikipedia.org/wiki/Maharashtra"
html = urllib.request.urlopen(url).read()
soup = BeautifulSoup(html, 'html.parser')

# Get text from paragraphs to keep it relevant
text = " ".join([p.text for p in soup.find_all('p')])

# Tokenize and POS Tagging
words = word_tokenize(text)
pos_tags = nltk.pos_tag(words)

# 1. Chunking for <VBD><DT>
print("\n1. Chunks of type <VBD><DT>:")
pattern = "Chunk: {<VBD><DT>}"
chunk_parser = nltk.RegexpParser(pattern)
chunked_tree = chunk_parser.parse(pos_tags)
for subtree in chunked_tree.subtrees():
    if subtree.label() == 'Chunk':
        print(subtree.leaves())

# 2. Find all Named Entities
print("\n2. Named Entities (First 10 shown for brevity):")
ne_tree = nltk.ne_chunk(pos_tags)
ne_count = 0
for child in ne_tree:
    if hasattr(child, 'label'):
        print(f"{child.label()}: {' '.join(c[0] for c in child)}")
        ne_count += 1
        if ne_count >= 10: 
            break

# 3. Porter and Lancaster Stemming on Verbs
print("\n3. Stemming on first 5 verbs:")
ps = PorterStemmer()
ls = LancasterStemmer()
verb_count = 0
for word, tag in pos_tags:
    if tag.startswith('VB'):
        print(f"Verb: {word} | Porter: {ps.stem(word)} | Lancaster: {ls.stem(word)}")
        verb_count += 1
        if verb_count >= 5: 
            break

# 4. Lemmatize unique past tense verbs
print("\n4. Unique Lemmatized Past Tense Verbs (First 10):")
lemmatizer = WordNetLemmatizer()
past_verbs = set()
for word, tag in pos_tags:
    if tag == 'VBD':  # VBD is past tense verb
        past_verbs.add(lemmatizer.lemmatize(word.lower(), pos='v'))
print(list(past_verbs)[:10])

# 5. Synonyms of Adjectives
print("\n5. Synonyms of first 5 Adjectives:")
adj_count = 0
for word, tag in pos_tags:
    if tag.startswith('JJ'):  # JJ represents Adjectives
        synonyms = set()
        for syn in wordnet.synsets(word, pos=wordnet.ADJ):
            for lm in syn.lemmas():
                synonyms.add(lm.name())
        print(f"Adjective: {word} -> Synonyms: {list(synonyms)[:3]}")
        adj_count += 1
        if adj_count >= 5: 
            break

# 6. Antonyms of Verbs
print("\n6. Antonyms of Verbs (Showing words that have antonyms):")
ant_count = 0
for word, tag in pos_tags:
    if tag.startswith('VB'):
        antonyms = set()
        for syn in wordnet.synsets(word, pos=wordnet.VERB):
            for lm in syn.lemmas():
                if lm.antonyms():
                    antonyms.add(lm.antonyms()[0].name())
        if antonyms:
            print(f"Verb: {word} -> Antonyms: {list(antonyms)}")
            ant_count += 1
            if ant_count >= 5: 
                break

# 7 & 8. DATE, TIME, and GPE (Locations) using Named Entity Tree
print("\n7 & 8. Dates, Times, and Locations (GPE):")
dates_times = set()
locations = set()

# NLTK ne_chunk labels locations as 'GPE' (Geo-Political Entity) or 'FACILITY'
# Note: NLTK basic chunker doesn't tag DATE/TIME well, so we look for common patterns or GPE labels.
for child in ne_tree:
    if hasattr(child, 'label'):
        entity_name = ' '.join(c[0] for c in child)
        if child.label() == 'GPE':
            locations.add(entity_name)

print(f"Locations (First 10): {list(locations)[:10]}")
print("Note: Basic NLTK chunker doesn't natively tag 'DATE'/'TIME' tokens explicitly out of the box (requires Spacy).")

[nltk_data] Downloading package punkt to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\PGCP-AI\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping chunkers\maxent_ne_chunker.zip.
[nltk_data] Downloading package words to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...


--- Question 1 ---


[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\PGCP-
[nltk_data]     AI\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
